# Roteiro de Aula: Aprendizado Supervisionado e Não Supervisionado com imóveis

Este notebook orienta um laboratório integrado de Machine Learning usando `dataset_imoveis.csv`. A prática percorre classificação, regressão e agrupamento, relacionando os modelos aos conceitos teóricos da Unidade 3.

O conjunto de dados representa imóveis de diferentes regiões, tipos e características físicas. O objetivo é trabalhar um problema genérico de modelagem, sem depender de uma aplicação específica do mercado financeiro.

## Objetivos
- Diferenciar classificação, regressão e agrupamento.
- Explorar uma base realista com dados numéricos, categóricos e ausentes.
- Construir pré-processamento sem vazamento de dados.
- Comparar modelos de classificação e regressão.
- Interpretar underfitting, overfitting e o efeito da complexidade da árvore.
- Usar K-Means, agrupamento hierárquico e DBSCAN.
- Avaliar modelos supervisionados e a qualidade dos clusters.

## 1. Carregamento e inspeção inicial
1. Importe `pandas`, `numpy`, `matplotlib` e `seaborn`.
2. Carregue `dataset_imoveis.csv` em `df`.
3. Exiba dimensões, tipos, primeiras linhas e quantidade de nulos.
4. Identifique as variáveis que podem ser usadas como atributos e como alvos.

### Perguntas
- Qual é a diferença entre `preco_reais` e `faixa_preco`?
- Quais variáveis são numéricas e quais são categóricas?
- Por que não devemos usar o preço como entrada ao prever a faixa de preço?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
df = pd.read_csv('dataset_imoveis.csv')

# Inspecao inicial
print('Shape:', df.shape)
display(df.head())
print('Tipos:')
print(df.dtypes)
print('Nulos:')
display(df.isnull().sum().sort_values(ascending=False))

## 2. Análise exploratória
1. Examine estatísticas descritivas das variáveis numéricas.
2. Observe a distribuição do preço e a frequência das faixas.
3. Visualize relações entre área, distância do centro e preço.
4. Verifique se há indícios de classes desbalanceadas.

### Perguntas
- A variável `preco_reais` parece ter valores extremos?
- A distribuição das faixas é equilibrada?
- Uma relação visual forte é suficiente para afirmar que uma variável é causal?

In [ ]:
display(df.describe(include='all').T)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(data=df, x='preco_reais', kde=True, ax=axes[0])
sns.countplot(data=df, x='faixa_preco', ax=axes[1])
sns.scatterplot(data=df, x='area_m2', y='preco_reais', hue='faixa_preco', ax=axes[2])
plt.tight_layout()
plt.show()

## 3. Preparação das tarefas e divisão dos dados
Serão construídas três tarefas:

- **Classificação:** prever `faixa_preco`.
- **Regressão:** prever `preco_reais`.
- **Agrupamento:** encontrar perfis de imóveis sem usar rótulos.

Para classificação e regressão, remova os registros sem alvo. Divida treino e teste antes de ajustar imputadores, codificadores ou escaladores.

In [ ]:
from sklearn.model_selection import train_test_split

features = [
    'regiao', 'tipo_imovel', 'area_m2', 'quartos',
    'vagas_garagem', 'idade_imovel_anos', 'distancia_centro_km'
]

df_class = df.dropna(subset=['faixa_preco']).copy()
df_reg = df.dropna(subset=['preco_reais']).copy()

X_class = df_class[features]
y_class = df_class['faixa_preco']
X_reg = df_reg[features]
y_reg = df_reg['preco_reais']

# TODO: faca as divisoes treino/teste. Use stratify na classificacao.
# X_class_train, X_class_test, y_class_train, y_class_test = ...
# X_reg_train, X_reg_test, y_reg_train, y_reg_test = ...

print('Registros de classificacao:', len(df_class))
print('Registros de regressao:', len(df_reg))

## 4. EDA automatizada com `ydata-profiling`

1. Instale `ydata-profiling` no ambiente do notebook, se necessário.
2. Gere um relatório exploratório usando somente o conjunto de treino da classificação.
3. Examine estatísticas, valores ausentes, distribuições, correlações e alertas de qualidade.
4. Não use o conjunto de teste para decidir imputação, encoding, escala ou modelo.

### Documentação
- [Documentação do ydata-profiling](https://docs.profiling.ydata.ai/latest/)
- [ProfileReport](https://docs.profiling.ydata.ai/latest/features/advanced_usage/)

### Perguntas
- Por que o relatório deve ser gerado a partir do treino em uma tarefa supervisionada?
- Quais achados do relatório podem orientar o pré-processamento?
- Um alerta de correlação implica que uma variável deve ser removida automaticamente?

In [ ]:
# Instale o pacote no ambiente do notebook, se necessario:
# %pip install ydata-profiling
from ydata_profiling import ProfileReport

# O profiling supervisionado deve usar apenas dados de treino.
# df_class_train = X_class_train.copy()
# df_class_train['faixa_preco'] = y_class_train
# profile = ProfileReport(df_class_train, title='EDA - treino de classificacao', minimal=True)
# profile.to_file('relatorio_eda_treino.html')
# profile.to_notebook_iframe()

print('Gere o relatorio somente com df_class_train e revise seus alertas.')

## 5. Pré-processamento sem vazamento
1. Crie uma funcao que devolva um novo `ColumnTransformer` para cada tarefa.
2. Ajuste o pre-processador de classificacao somente em `X_class_train`.
3. Ajuste o pre-processador de regressao somente em `X_reg_train`.
4. Transforme os respectivos conjuntos de teste sem chamar `fit` neles.
5. Para agrupamento exploratorio, use um pre-processador separado e deixe claro que ele nao participa da avaliacao supervisionada.

### Checklist de boas práticas
- A remocao de linhas sem alvo ocorre antes da divisao, pois nao e imputacao de atributo.
- Nenhuma estatistica do teste deve definir mediana, moda, categorias ou escala.
- O conjunto de teste deve ser usado somente para medir o desempenho final.
- Nao use `preco_reais` nem `faixa_preco` como atributo de classificacao ou agrupamento.

### Perguntas
- Por que usar objetos de pre-processamento separados evita reutilizar estado entre tarefas?
- Qual e a diferenca entre `fit_transform` no treino e `transform` no teste?
- Por que o pre-processamento do agrupamento nao deve ser avaliado com acuracia?

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = ['area_m2', 'quartos', 'vagas_garagem', 'idade_imovel_anos', 'distancia_centro_km']
categorical_features = ['regiao', 'tipo_imovel']

def build_preprocessor():
    numeric_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    categorical_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    return ColumnTransformer([
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Cada tarefa recebe um pre-processador independente.
preprocessor_class = build_preprocessor()
preprocessor_reg = build_preprocessor()

# TODO: ajuste cada objeto exclusivamente no respectivo conjunto de treino.
# X_class_train_prep = preprocessor_class.fit_transform(X_class_train)
# X_class_test_prep = preprocessor_class.transform(X_class_test)
# X_reg_train_prep = preprocessor_reg.fit_transform(X_reg_train)
# X_reg_test_prep = preprocessor_reg.transform(X_reg_test)

print('Use fit_transform somente no treino e transform somente no teste.')

## 5. Classificação: comparação de modelos
Compare pelo menos três modelos:

- Regressão Logística;
- KNN;
- Árvore de Decisão.

Calcule acurácia e F1 macro. Para a árvore, experimente diferentes valores de `max_depth`. Exiba a matriz de confusão do melhor modelo.

### Perguntas
- O modelo com maior acurácia também possui maior F1 macro?
- O que uma árvore muito profunda pode indicar?
- Como o valor de `k` afeta o KNN?

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

# TODO: crie os modelos e treine cada um com os dados preparados.
# modelos_class = {...}
# resultados_class = []
# para cada modelo, calcule accuracy e f1_macro

print('Compare os modelos e escolha o melhor com justificativa.')

## 6. Regressão: linearidade e regularização
Compare Regressão Linear, Ridge e Lasso. Use MAE, RMSE e R².

### Perguntas
- Qual modelo apresenta menor erro no teste?
- A regularização mudou o desempenho ou a estabilidade?
- Um R² alto significa necessariamente que o modelo é útil?

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# TODO: prepare os dados de regressao com o mesmo preprocessor, ajustado apenas no treino.
# X_reg_train_prep = ...
# X_reg_test_prep = ...

# TODO: compare LinearRegression, Ridge(alpha=1.0) e Lasso(alpha=0.01).
# Calcule MAE, RMSE e R2 para cada modelo.

print('Compare os modelos de regressao com as tres metricas.')

## 7. Regressão: complexidade da árvore
Treine árvores de decisão com `max_depth=3`, `max_depth=8` e sem limite. Compare os erros de treino e teste.

### Perguntas
- Em que situação a diferença entre treino e teste sugere overfitting?
- O que acontece quando a árvore fica mais complexa?
- Qual profundidade parece produzir melhor equilíbrio?

In [ ]:
from sklearn.tree import DecisionTreeRegressor

# TODO: treine as tres arvores e registre MAE de treino e teste.
# profundidades = [3, 8, None]
# ...

print('Analise a diferenca entre erro de treino e erro de teste.')

## 8. Agrupamento exploratório
Agora ignore `preco_reais` e `faixa_preco`. O objetivo é descobrir perfis de imóveis apenas a partir de suas características.

1. Prepare os dados com imputação, encoding e escala.
2. Use o método do cotovelo para testar `k` entre 2 e 6.
3. Use a silhueta para apoiar a escolha de `k`.
4. Treine K-Means e visualize os grupos com PCA.

### Perguntas
- O cotovelo e a silhueta indicam o mesmo valor de `k`?
- Como interpretar um valor baixo de silhueta?
- O preço pode ser usado depois apenas para descrever os clusters, sem ter participado da formação?

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

X_cluster = df[features]
# TODO: crie um preprocessor para X_cluster e transforme os dados.
# X_cluster_prep = ...

# TODO: calcule inertia e silhouette para k de 2 a 6.
# escolha um k e treine o modelo final.

# TODO: reduza para duas componentes com PCA e faça um scatterplot colorido pelos clusters.
print('Use o cotovelo e a silhueta para justificar o k escolhido.')

## 9. Agrupamento hierárquico e DBSCAN
1. Compare o agrupamento hierárquico com o K-Means usando o mesmo `k`.
2. Calcule a silhueta dos dois agrupamentos.
3. Teste DBSCAN com valores diferentes de `eps` e `min_samples`.
4. Conte clusters encontrados e pontos classificados como ruído.

### Perguntas
- Por que o DBSCAN não exige informar `k`?
- O que significa o rótulo `-1`?
- Qual método parece mais adequado à estrutura desta base? Justifique com métricas e visualizações.

In [ ]:
from sklearn.cluster import AgglomerativeClustering, DBSCAN

# TODO: treine AgglomerativeClustering com o k escolhido.
# TODO: calcule a silhueta do agrupamento hierarquico.
# TODO: treine DBSCAN e avalie clusters e ruido.

print('Compare K-Means, hierarquico e DBSCAN.')

## 10. Visualização final dos clusters com PCA

O PCA (Principal Component Analysis) transforma várias variáveis correlacionadas em novas componentes lineares. A primeira componente captura a maior variância possível, a segunda captura a maior parte restante e assim por diante. A visualização em duas componentes é uma aproximação: ela facilita observar padrões, mas pode esconder informação das dimensões descartadas.

Acesse a documentação antes de executar:
- [Documentação do PCA no scikit-learn](https://scikit-learn.org/stable/modules/decomposition.html#pca)
- [Exemplo oficial de redução de dimensionalidade](https://scikit-learn.org/stable/auto_examples/decomposition/plot_pca_iris.html)

### Perguntas
- O que significa reduzir a dimensionalidade de 13 atributos para 2 componentes?
- Qual proporção da variância é explicada pelas duas componentes?
- Um cluster separado no gráfico 2D necessariamente está bem separado no espaço original?

In [ ]:
from sklearn.decomposition import PCA

# O PCA e usado aqui para visualizacao, nao para treinar os modelos.
# TODO: aplique PCA(n_components=2) em X_cluster_prep.
# pca = PCA(n_components=2, random_state=42)
# X_cluster_pca = pca.fit_transform(X_cluster_prep)
# print('Variancia explicada:', pca.explained_variance_ratio_)
# sns.scatterplot(x=X_cluster_pca[:, 0], y=X_cluster_pca[:, 1], hue=kmeans_labels, palette='viridis')
# plt.title('Clusters K-Means visualizados com PCA')
# plt.xlabel('Componente principal 1')
# plt.ylabel('Componente principal 2')
# plt.show()

print('Relacione a visualizacao com a variancia explicada pelas componentes.')

## 11. Conclusão
Registre:
- o melhor modelo de classificação e a métrica usada para escolhê-lo;
- o melhor modelo de regressão e seus erros;
- sinais de underfitting ou overfitting;
- o valor de `k` escolhido e a silhueta;
- as diferenças entre K-Means, hierárquico e DBSCAN;
- limitações e possíveis melhorias da análise.